In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from plotly.offline import iplot

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
import keras 
from keras.models import Sequential
from keras.optimizers import Adam, Adamax
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.layers import MaxPooling2D, Flatten, Dense,BatchNormalization,GlobalAveragePooling2D,Conv2D,Dropout,Flatten,Rescaling,Input
from keras import regularizers
from keras.callbacks import EarlyStopping,ModelCheckpoint

import os
import random
import cv2

c:\Users\saulo\PIBIC\TinyML-Robustness-Mango\.venv\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning:

Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.

c:\Users\saulo\PIBIC\TinyML-Robustness-Mango\.venv\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning:

Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.

c:\Users\saulo\PIBIC\TinyML-Robustness-Mango\.venv\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning:

Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/resource_handle.proto. Please update t

In [ ]:
data_path = "../../MangoLeaf"
img_size = (224,224)
batch_size = 32
mode = "rgb"
epochs = 50

In [ ]:
def df_maker(path):
    file_paths = []
    labels = []

    folds = os.listdir(path)
    for fold in folds:
        fold_path = os.path.join(path,fold)
        file_list = os.listdir(fold_path)
        for file in file_list:
            file_path = os.path.join(fold_path,file)
            file_paths.append(file_path)
            labels.append(fold)


    file_series = pd.Series(file_paths,name="file_paths")
    label_series = pd.Series(labels,name="labels")

    df = pd.concat([file_series,label_series],axis=1)
    return df

In [ ]:
df = df_maker(data_path)

train_df,test_val_df= train_test_split(df ,train_size= 0.8, shuffle= True, random_state= 7, stratify=df["labels"])

test_df,val_df= train_test_split(test_val_df ,train_size= 0.5, shuffle= True, random_state= 7, stratify=test_val_df["labels"])

In [ ]:
from tensorflow.keras.applications import EfficientNetB0


In [ ]:
# Preparar geradores de dados com preprocessamento do EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input


# Classes e parâmetros
class_names = sorted(df['labels'].unique())
num_classes = len(class_names)
print(f"Classes ({num_classes}):", class_names)


train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.10,
    horizontal_flip=True,
    fill_mode='nearest'
 )
val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)


common_args = dict(
    x_col='file_paths',
    y_col='labels',
    target_size=img_size,
    color_mode='rgb',
    class_mode='categorical',
    batch_size=batch_size,
    shuffle=True,
    seed=42,
    validate_filenames=False
)


train_gen = train_datagen.flow_from_dataframe(train_df, directory=None, **common_args)
val_gen = val_datagen.flow_from_dataframe(val_df, directory=None, **{**common_args, 'shuffle': False})
test_gen = test_datagen.flow_from_dataframe(test_df, directory=None, **{**common_args, 'shuffle': False})

In [ ]:
# Construir o modelo com EfficientNetB0 como base
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0


with tf.device('/GPU:0') if tf.config.list_physical_devices('GPU') else tf.device('/CPU:0'):
    base_model = EfficientNetB0(include_top=False, weights='imagenet', input_shape=img_size + (3,))
    base_model.trainable = False  # congela no início para treinar topo primeiro


    inputs = layers.Input(shape=img_size + (3,))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    model = models.Model(inputs, outputs)


opt = Adam(learning_rate=1e-3)
model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# Callbacks e treino do topo (cabea)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau


checkpoint_path = '../../v2/models/efficientnet_b0_mangoleaf.keras'
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6),
    ModelCheckpoint(checkpoint_path, monitor='val_accuracy', save_best_only=True, verbose=1)
]

steps_per_epoch = max(1, len(train_df)//batch_size)
validation_steps = max(1, len(val_df)//batch_size)


history = model.fit(
    train_gen,
    epochs=epochs,
    validation_data=val_gen,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# Fine-tuning: descongelar as ltimas camadas da base e treinar com LR baixo
unfreeze_from = int(0.75 * len(base_model.layers))
for i, layer in enumerate(base_model.layers):
    layer.trainable = (i >= unfreeze_from)
print(f"Descongelando a partir da camada {unfreeze_from}/{len(base_model.layers)}")


opt_ft = Adam(learning_rate=1e-4)
model.compile(optimizer=opt_ft, loss='categorical_crossentropy', metrics=['accuracy'])


history_ft = model.fit(
    train_gen,
    epochs=max(10, epochs//3),
    validation_data=val_gen,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    callbacks=callbacks,
    verbose=1
)

## Modelo EfficientNetB0 para MangoLeaf

A célula abaixo cria um modelo baseado no EfficientNetB0 (topo removido), adiciona uma cabeça leve para classificar as classes do MangoLeaf e compila o modelo pronto para treino.

In [ ]:
# Criação do modelo EfficientNetB0 para MangoLeaf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0

# Garante que num_classes e img_size existam
try:
    _ = num_classes
except NameError:
    class_names = sorted(df['labels'].unique())
    num_classes = len(class_names)

input_shape = tuple(img_size) + (3,)

with tf.device('/GPU:0') if tf.config.list_physical_devices('GPU') else tf.device('/CPU:0'):
    base_model = EfficientNetB0(include_top=False, weights='imagenet', input_shape=input_shape)
    base_model.trainable = False  # treinar apenas a cabeça primeiro

    inputs = layers.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    model = models.Model(inputs, outputs)

# Compilação
model.compile(optimizer=Adam(learning_rate=1e-3),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()